# Joining, Reshaping & Missing Data
Tools: Python, pandas, VS Code and/or Jupyter

## Learning objectives
By the end of this tutorial, you should be able to:

combine tables using merge();

distinguish inner, left, right, and outer joins;

identify and diagnose unmatched records;

detect duplicate keys and validate join cardinality;

combine datasets using concat();

convert data between wide and long form;

use melt(), pivot(), and pivot_table();

detect and summarize missing data;

apply simple missing-data strategies critically;

validate a multi-step wrangling pipeline.

In [1]:
import pandas as pd
import numpy as np

In [2]:
participants = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106],
    "age": [24, 31, 27, 19, 42, 35],
    "group": ["A", "B", "A", "B", "A", "B"]
})

In [3]:
scores = pd.DataFrame({
    "id": [101, 102, 103, 105, 106, 107],
    "score": [81, 94, 88, 91, 85, 79]
})

In [4]:
print(f"{participants =}")
print(f"{scores =}")

participants =    id  age group
0  101   24     A
1  102   31     B
2  103   27     A
3  104   19     B
4  105   42     A
5  106   35     B
scores =    id  score
0  101     81
1  102     94
2  103     88
3  105     91
4  106     85
5  107     79


**Questions**
1. Which IDs occur in both tables?

    101, 102, 103, 105, and 106

2. Which ID occurs only in participants?

    104

3. Which ID occurs only in scores?

    107

4. What do you expect an inner join to contain?

    Only the IDs in both tables: 101, 102, 103, 105, and 106

In [5]:
inner = participants.merge(
    scores,
    on="id",
    how="inner"
)

print(f"{inner = }")
print(f"{inner.shape = }")

inner =     id  age group  score
0  101   24     A     81
1  102   31     B     94
2  103   27     A     88
3  105   42     A     91
4  106   35     B     85
inner.shape = (5, 4)


**Which IDs were removed?**

104 and 107

**Why?**

They belonged to one table but not both.

**Did the number of rows match your prediction?**

Yes


In [6]:
left = participants.merge(
    scores,
    on="id",
    how="left"
)

print(left)

    id  age group  score
0  101   24     A   81.0
1  102   31     B   94.0
2  103   27     A   88.0
3  104   19     B    NaN
4  105   42     A   91.0
5  106   35     B   85.0


**Which table defines the resulting population?**
The participants table; there is no 107

**What happened to ID 104?**
The ID has no observation in the scores table, so it got a NAN value.

**Why is its score missing?**
See above

In [7]:
outer = participants.merge(
    scores,
    on="id",
    how="outer"
)

print(outer)

    id   age group  score
0  101  24.0     A   81.0
1  102  31.0     B   94.0
2  103  27.0     A   88.0
3  104  19.0     B    NaN
4  105  42.0     A   91.0
5  106  35.0     B   85.0
6  107   NaN   NaN   79.0


A right join will contain all IDs except for 104.

In [9]:
right = participants.merge(
    scores,
    on="id",
    how="right"
)

print(right)

    id   age group  score
0  101  24.0     A     81
1  102  31.0     B     94
2  103  27.0     A     88
3  105  42.0     A     91
4  106  35.0     B     85
5  107   NaN   NaN     79


### Concatenate Rows

In [11]:
fall = pd.DataFrame({
    "id": [1, 2, 3],
    "semester": ["Fall"] * 3,
    "score": [80, 85, 90]
})

spring = pd.DataFrame({
    "id": [4, 5, 6],
    "semester": ["Spring"] * 3,
    "score": [78, 92, 88]
})

In [13]:
all_scores = pd.concat(
    [fall, spring],
    axis=0,
    ignore_index=True
)

all_scores

,id,semester,score
0,1,Fall,80
1,2,Fall,85
2,3,Fall,90
3,4,Spring,78
4,5,Spring,92
5,6,Spring,88


`concat()` is more natural than `merge()` here because we are adding two discrete sets without overlapping keys together into one dataframe.

In [15]:
fall = pd.DataFrame({
    "id": [1, 2],
    "score": [80, 85]
})

spring = pd.DataFrame({
    "id": [3, 4],
    "score": [90, 92],
    "campus": ["Flagstaff", "Flagstaff"]
})

pd.concat(
    [fall, spring],
    ignore_index=True
)

,id,score,campus
0,1,80,NaN
1,2,85,NaN
2,3,90,Flagstaff
3,4,92,Flagstaff


**What does pandas do with the column that does not exist in fall?**

It fills in the missing values for the nonexistant column with NaN (none values).

## Wide Data

In [18]:
wide = pd.DataFrame({
    "id": [101, 102, 103, 104],
    "group": ["A", "B", "A", "B"],
    "score_pre": [72, 85, 79, 88],
    "score_post": [81, 91, 84, 90]
})

wide

,id,group,score_pre,score_post
0,101,A,72,81
1,102,B,85,91
2,103,A,79,84
3,104,B,88,90


**How many rows represent each participant?**

1 with 2 observations per row

**How are repeated measurements represented?**

As a new column (`score_pre` and `score_post`)

In [19]:
long = wide.melt(
    id_vars=["id", "group"],
    value_vars=["score_pre", "score_post"],
    var_name="time",
    value_name="score"
)

long

,id,group,time,score
0,101,A,score_pre,72
1,102,B,score_pre,85
2,103,A,score_pre,79
3,104,B,score_pre,88
4,101,A,score_post,81
5,102,B,score_post,91
6,103,A,score_post,84
7,104,B,score_post,90


**How many rows now represent each participant?**

2 rows per participant

**What happened to the original score columns?**

Their score values became part of the same score column and a new column `time` was added to indicate which score is which.

**What does id_vars mean?**

`id_vars` are the variables that uniquely identify each participant/observation.

**What does value_vars mean?**

`value_vars` are the variables with the values measured to be included in the "value" column `score`.

In [21]:
long["time"] = (
    long["time"]
    .str.replace("score_", "", regex=False)
)

long

,id,group,time,score
0,101,A,pre,72
1,102,B,pre,85
2,103,A,pre,79
3,104,B,pre,88
4,101,A,post,81
5,102,B,post,91
6,103,A,post,84
7,104,B,post,90


In [24]:
# This works when run on the frame after it is cleaned by the previous cell
long["time"] = (
    long["time"]
    .str.replace("pre", "baseline", regex=False).replace("post", "followup")
)

long

,id,group,time,score
0,101,A,baseline,72
1,102,B,baseline,85
2,103,A,baseline,79
3,104,B,baseline,88
4,101,A,followup,81
5,102,B,followup,91
6,103,A,followup,84
7,104,B,followup,90


## Long to Wide

In [25]:
wide_again = long.pivot(
    index=["id", "group"],
    columns="time",
    values="score"
)

wide_again

,time,baseline,followup
id,group,,
101,A,72,81
102,B,85,91
103,A,79,84
104,B,88,90


In [26]:
wide_again = wide_again.reset_index()
wide_again

time,id,group,baseline,followup
0,101,A,72,81
1,102,B,85,91
2,103,A,79,84
3,104,B,88,90


The resulting dataframe has the same information, but the columns `score_pre` and `score_post` now have the new names that replaced them when cleaning the long data.

## Duplicate Combinations and `pivot()`

In [31]:
repeated = pd.DataFrame({
    "id": [1, 1, 1, 2, 2],
    "time": ["pre", "pre", "post", "pre", "post"],
    "score": [70, 74, 82, 80, 88]
})
repeated

,id,time,score
0,1,pre,70
1,1,pre,74
2,1,post,82
3,2,pre,80
4,2,post,88


In [28]:
repeated.pivot(
    index="id", # Duplicate entries here
    columns="time",
    values="score"
)

ValueError: Index contains duplicate entries, cannot reshape

The above cell does not work because there are duplicate identifier entries that the code would collapse if pivoted. I assume it throws an error to protect data from being collapsed. It violates the assumption that each index × column pair is unique. `ID` and `time` both cary repeats.

In [30]:
summary = repeated.pivot_table(
    index="id",
    columns="time",
    values="score",
    aggfunc="mean"
)
summary

time,post,pre
id,,
1,82.0,72.0
2,88.0,80.0


**Why should you investigate duplicate records before deciding that taking the mean is appropriate?**

Repeated observations of the same thing represent duplicate entries that can skew the mean. The true value for ID 1 pre should be either 70 or 74, depending on the "correct" observation.


## Missing Data

In [50]:
clinical = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "age": [24, 31, np.nan, 45, 28, 39],
    "group": ["A", "B", "A", "B", None, "A"],
    "score": [81, np.nan, 77, 92, 88, np.nan]
})

clinical

,id,age,group,score
0,1,24.0,A,81.0
1,2,31.0,B,NaN
2,3,NaN,A,77.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0
5,6,39.0,A,NaN


In [35]:
print(f"Is NA:{clinical.isna()}\n")
print(f"Number of NA: {clinical.isna().sum()}\n")
print(f"% of values Missing: {clinical.isna().mean() * 100}\n")

Is NA:      id    age  group  score
0  False  False  False  False
1  False  False  False   True
2  False   True  False  False
3  False  False  False  False
4  False  False   True  False
5  False  False  False   True

Number of NA: id       0
age      1
group    1
score    2
dtype: int64

% of values Missing: id        0.000000
age      16.666667
group    16.666667
score    33.333333
dtype: float64



In [36]:
# Find scores with missing data
clinical.loc[
    clinical["score"].isna()
]

,id,age,group,score
1,2,31.0,B,NaN
5,6,39.0,A,NaN


In [37]:
# Find scores with NOT missing data
clinical.loc[
    clinical["score"].notna()
]

,id,age,group,score
0,1,24.0,A,81.0
2,3,NaN,A,77.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0


In [39]:
# Where age OR score is missing
clinical.loc[
    clinical[["age", "score"]]
    .isna()
    .any(axis=1)
]

,id,age,group,score
1,2,31.0,B,NaN
2,3,NaN,A,77.0
5,6,39.0,A,NaN


In [52]:
clinical.loc[
    clinical[["age", "score"]]
    .notna()
    .all(axis=1)
]

,id,age,group,score
0,1,24.0,A,81.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0


## Drop missing

In [59]:
complete_score = clinical.dropna(
    subset=["score"]
)
print(complete_score)
print(f"\n{len(clinical) = }\n{len(complete_score)= }")

   id   age group  score
0   1  24.0     A   81.0
2   3   NaN     A   77.0
3   4  45.0     B   92.0
4   5  28.0   NaN   88.0

len(clinical) = 6
len(complete_score)= 4


In [58]:
complete_cases = clinical.dropna(
    subset=["age", "score"]
)

complete_cases

,id,age,group,score
0,1,24.0,A,81.0
3,4,45.0,B,92.0
4,5,28.0,NaN,88.0


**How many observations were removed?**
3 observations

**Has the analysis population changed?**

no


**What assumptions might make complete-case analysis problematic?**

There might be a systematic reason values are missing. Dropping the NA values might remove a particular group that is present in the population, but now no longer represented in the sample.

## Filling mising values

In [63]:
filled = clinical.copy()
filled["score"] = filled["score"].fillna(
    filled["score"].mean()
)

print(f"{clinical["score"].describe() = }\n")
print(f"{filled["score"].describe() = }")

clinical["score"].describe() = count     4.000000
mean     84.500000
std       6.757712
min      77.000000
25%      80.000000
50%      84.500000
75%      89.000000
max      92.000000
Name: score, dtype: float64

filled["score"].describe() = count     6.000000
mean     84.500000
std       5.234501
min      77.000000
25%      81.875000
50%      84.500000
75%      87.125000
max      92.000000
Name: score, dtype: float64


**Did the mean change?**

Nope

**Did the standard deviation change?**

Yes

**Why?**

The imputed values were filled with the mean, so the values which were previously empty are now right on top of the mean. This will reduce the variance by having more values closer to (in this case right at) the mean.

**Is mean imputation automatically a good statistical strategy?**

No. It might hide important variance, especially among groups in the data.

## Missingness introduced by a join

In [66]:
joined = participants.merge(
    scores,
    on="id",
    how="left"
)
joined

,id,age,group,score
0,101,24,A,81.0
1,102,31,B,94.0
2,103,27,A,88.0
3,104,19,B,NaN
4,105,42,A,91.0
5,106,35,B,85.0


In [65]:
joined.loc[
    joined["score"].isna()
]

,id,age,group,score
3,104,19,B,NaN


**Does a missing score here necessarily mean that the participant failed to provide a score? What other explanation exists?**

No, it may be that the value was deleted from the data, the participant took a different test, or the test was invalidated, among other possible explanations. The participant could also have been ineligible to take the test, maybe due to an age restriction.

## Missingness indicator

In [67]:
clinical["score_missing"] = (
    clinical["score"].isna()
)

clinical

,id,age,group,score,score_missing
0,1,24.0,A,81.0,False
1,2,31.0,B,NaN,True
2,3,NaN,A,77.0,False
3,4,45.0,B,92.0,False
4,5,28.0,NaN,88.0,False
5,6,39.0,A,NaN,True


In [68]:
pd.crosstab(
    clinical["group"],
    clinical["score_missing"]
)

score_missing,False,True
group,,
A,2,1
B,1,1


**What question does this table help you investigate?**

This table helps see which group might be over/underrepresented in the data based on missingness or if one group has has more missing scores to find systematic issues in the data.

## Complete workflow

In [76]:
demographics = pd.DataFrame({
    "id": [101, 102, 103, 104, 105],
    "age": [24, 31, 27, 42, 36],
    "group": ["A", "B", "A", "B", "A"]
})

outcomes = pd.DataFrame({
    "id": [101, 102, 103, 105],
    "pre": [72, 80, 85, 78],
    "post": [81, 87, np.nan, 86]
})

Produce a long format dataset with 

id

age

group

time

score

In [70]:
print(f"{demographics["id"].is_unique = }\n")
print(f"{outcomes["id"].is_unique = }\n")
print(f"{set(demographics["id"]) - set(outcomes["id"]) = }\n")
print(f"{set(outcomes["id"]) - set(demographics["id"]) = }\n")

demographics["id"].is_unique = True

outcomes["id"].is_unique = True

set(demographics["id"]) - set(outcomes["id"]) = {104}

set(outcomes["id"]) - set(demographics["id"]) = set()



A merge based on the IDs will either remove ID 104 or fill in the outcomes values for that ID with NAs

In [77]:
combined = demographics.merge(
    outcomes,
    on="id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print(f"{combined = }\n")
print(f"{combined["_merge"].value_counts() = }")

combined =     id  age group   pre  post     _merge
0  101   24     A  72.0  81.0       both
1  102   31     B  80.0  87.0       both
2  103   27     A  85.0   NaN       both
3  104   42     B   NaN   NaN  left_only
4  105   36     A  78.0  86.0       both

combined["_merge"].value_counts() = _merge
both          4
left_only     1
right_only    0
Name: count, dtype: int64


In [78]:
combined = combined.drop(
    columns="_merge"
)

combined

,id,age,group,pre,post
0,101,24,A,72.0,81.0
1,102,31,B,80.0,87.0
2,103,27,A,85.0,NaN
3,104,42,B,NaN,NaN
4,105,36,A,78.0,86.0


In [79]:
analysis = combined.melt(
    id_vars=["id", "age", "group"],
    value_vars=["pre", "post"],
    var_name="time",
    value_name="score"
)

analysis

,id,age,group,time,score
0,101,24,A,pre,72.0
1,102,31,B,pre,80.0
2,103,27,A,pre,85.0
3,104,42,B,pre,NaN
4,105,36,A,pre,78.0
5,101,24,A,post,81.0
6,102,31,B,post,87.0
7,103,27,A,post,NaN
8,104,42,B,post,NaN
9,105,36,A,post,86.0


In [80]:
analysis.isna().sum()

id       0
age      0
group    0
time     0
score    3
dtype: int64

In [81]:
analysis.loc[
    analysis["score"].isna()
]

,id,age,group,time,score
3,104,42,B,pre,NaN
7,103,27,A,post,NaN
8,104,42,B,post,NaN


For each missing score, determine whether it arose because:
* an outcome record was absent entirely; or
* an outcome row existed but a particular measurement was missing.
These are different data problems.

The missing pre and post scores for ID 104 are because that ID is entirely absent in the outcomes dataset. The missing post score for ID 103 had a missing measurement.

In [83]:
observed = analysis.dropna(
    subset=["score"]
)

n_before = len(analysis)
n_after = len(observed)
n_removed = n_before - n_after

analysis_report = {
"n_before" :n_before,
"n_after" :n_after,
"n_removed" :n_removed}

print(analysis_report)


{'n_before': 10, 'n_after': 7, 'n_removed': 3}


**Why should this information appear in an analysis report?**

These values show how big the analysed sample is and how much data in the set is complete.

In [86]:
print(f"{analysis["id"].nunique() = }\n" )
print(f"{demographics["id"].nunique() = }\n") 
print(f"{analysis["time"].value_counts() = }\n" )

analysis["id"].nunique() = 5

demographics["id"].nunique() = 5

analysis["time"].value_counts() = time
pre     5
post    5
Name: count, dtype: int64



In [88]:
analysis.duplicated(
    subset=["id", "time"]
).sum()

np.int64(0)

In [89]:
assert (
    analysis["id"].nunique()
    == demographics["id"].nunique()
)

assert (
    analysis.duplicated(
        subset=["id", "time"]
    ).sum()
    == 0
)

## Three mini datasets

In [90]:
jan = pd.DataFrame({
    "id": [1, 2],
    "month": ["Jan", "Jan"],
    "score": [80, 85]
})

feb = pd.DataFrame({
    "id": [1, 2],
    "month": ["Feb", "Feb"],
    "score": [82, 88]
})

mar = pd.DataFrame({
    "id": [1, 2],
    "month": ["Mar", "Mar"],
    "score": [84, np.nan]
})

Concatenate the three tables.

Verify the number of rows.

Identify missing scores.

Pivot into wide form with one row per participant.

Convert back to long form.

Verify that the structure is equivalent to the original combined data.

In [91]:
concat_months = pd.concat([jan, feb, mar])

concat_months

,id,month,score
0,1,Jan,80.0
1,2,Jan,85.0
0,1,Feb,82.0
1,2,Feb,88.0
0,1,Mar,84.0
1,2,Mar,NaN


In [94]:
print(f"nrow per month\n {len(jan) = }\n{len(feb) = }\n{len(mar) = }\n")

print(f"{len(concat_months) = }")

nrow per month
 len(jan) = 2
len(feb) = 2
len(mar) = 2

len(concat_months) = 6


In [93]:
concat_months.loc[concat_months["score"].isna()]

,id,month,score
1,2,Mar,NaN


In [104]:
wide_months = concat_months.pivot(
    index = ["id"],
    columns = "month",
    values = "score"
)

wide_months

month,Feb,Jan,Mar
id,,,
1,82.0,80.0,84.0
2,88.0,85.0,NaN


In [105]:
long_again_months = wide_months.melt(
    id_vars=["id", "month"],
    value_vars=["Feb", "Jan", "Mar"],
    var_name="month",
    value_name="score"
)

KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['id', 'month']"

long_again_months = wide_months.melt()